# Data Loading and Initial Setup

Load the raw e-commerce data from CSV files, set up the environment, and import necessary libraries such as pandas, numpy, matplotlib, and seaborn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Load data
df = pd.read_csv('../data/processed/cleaned_data.csv')
print("Data loaded successfully")
print("Shape:", df.shape)

# Data Cleaning and Preprocessing

Clean the data by handling missing values, duplicates, and outliers; convert data types; and prepare the dataset for analysis.

In [ ]:
# Assuming data is cleaned
df['order_date'] = pd.to_datetime(df['order_date'])

# Feature engineering for segmentation
customer_df = df.groupby('customer_unique_id').agg({
    'order_id': 'count',
    'total': ['sum', 'mean'],
    'price': 'mean',
    'order_date': ['min', 'max']
}).reset_index()

customer_df.columns = ['customer_unique_id', 'num_orders', 'total_revenue', 'avg_order_value', 'avg_price', 'first_order', 'last_order']

# Calculate recency, frequency, monetary (RFM)
current_date = df['order_date'].max()
customer_df['recency'] = (current_date - customer_df['last_order']).dt.days
customer_df['frequency'] = customer_df['num_orders']
customer_df['monetary'] = customer_df['total_revenue']

print("Customer features prepared:")
display(customer_df.head())

# Exploratory Data Analysis (EDA)

Perform EDA to understand the data distribution, key statistics, correlations, and visualize trends using plots and charts.

In [ ]:
# EDA on customer features
print("Customer feature statistics:")
display(customer_df[['recency', 'frequency', 'monetary']].describe())

# Distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.histplot(customer_df['recency'], ax=axes[0], kde=True)
axes[0].set_title('Recency Distribution')
sns.histplot(customer_df['frequency'], ax=axes[1], kde=True)
axes[1].set_title('Frequency Distribution')
sns.histplot(customer_df['monetary'], ax=axes[2], kde=True)
axes[2].set_title('Monetary Distribution')
plt.show()

# Correlation
plt.figure(figsize=(8, 6))
sns.heatmap(customer_df[['recency', 'frequency', 'monetary']].corr(), annot=True, cmap='coolwarm')
plt.title('RFM Correlation Matrix')
plt.show()

# Customer Segmentation

Segment customers using clustering techniques like K-means or RFM analysis to identify high-value groups.

In [ ]:
# Prepare data for clustering
rfm_features = customer_df[['recency', 'frequency', 'monetary']]

# Scale the features
scaler = StandardScaler()
scaled_rfm = scaler.fit_transform(rfm_features)

# Determine optimal number of clusters using elbow method
inertia = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_rfm)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(scaled_rfm, kmeans.labels_))

# Plot elbow curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')

plt.subplot(1, 2, 2)
plt.plot(k_range, silhouette_scores, marker='o')
plt.title('Silhouette Score')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.show()

# Apply K-means with optimal k (assuming 4 based on common practice)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customer_df['cluster'] = kmeans.fit_predict(scaled_rfm)

print("Customer segments created:")
display(customer_df.groupby('cluster')[['recency', 'frequency', 'monetary']].mean())

# Visualize clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(x='recency', y='monetary', hue='cluster', data=customer_df, palette='viridis')
plt.title('Customer Segments: Recency vs Monetary')
plt.show()

# Segment profiling
segment_profile = customer_df.groupby('cluster').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean',
    'customer_id': 'count'
}).rename(columns={'customer_id': 'count'})

print("Segment Profiles:")
display(segment_profile)

# Revenue Analysis and Insights

Analyze revenue streams, profitability, and generate key business insights with visualizations and summary statistics.

In [ ]:
# Revenue by segment
revenue_by_segment = customer_df.groupby('cluster')['monetary'].sum()

plt.figure(figsize=(8, 5))
revenue_by_segment.plot(kind='bar')
plt.title('Revenue by Customer Segment')
plt.ylabel('Total Revenue')
plt.show()

# Insights
total_revenue = customer_df['monetary'].sum()
top_segment = revenue_by_segment.idxmax()
top_segment_revenue = revenue_by_segment.max()
top_segment_percentage = (top_segment_revenue / total_revenue) * 100

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Top segment (Cluster {top_segment}): ${top_segment_revenue:,.2f} ({top_segment_percentage:.1f}% of total)")

# Recommendations
recommendations = [
    f"Focus marketing efforts on Cluster {top_segment} as they generate the most revenue.",
    "Implement retention strategies for high-recency customers to reduce churn.",
    "Create personalized offers for low-frequency customers to increase purchase frequency."
]

print("\nRecommendations:")
for rec in recommendations:
    print(f"- {rec}")

# Save segmented data
customer_df.to_csv('../data/processed/customer_segments.csv', index=False)
print("Segmented customer data saved.")